In [9]:
import os
import sqlite3
import chromadb
from chromadb.utils import embedding_functions

# ==========================================
# 1. Core Functions (From previous steps)
# ==========================================
def extract_sqlite_ddl(db_path):
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name, sql FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        
        ddl_statements = []
        for table_name, sql_statement in tables:
            if table_name != 'sqlite_sequence':
                ddl_statements.append(f"-- Table structure: {table_name}\n{sql_statement}")
                
        conn.close()
        return "\n\n".join(ddl_statements)
    except sqlite3.Error as e:
        return f"Error accessing database: {e}"

def setup_vector_database(db_id, db_ddl, schema_collection):
    # Notice we pass the collection as a parameter now to avoid reconnecting every loop
    schema_collection.add(
        documents=[db_ddl],
        metadatas=[{"db_id": db_id}],
        ids=[f"schema_{db_id}"]
    )
    print(f"Schema '{db_id}' vectorized successfully.")

# ==========================================
# 2. Batch Processing Automation
# ==========================================
def process_all_spider_databases(spider_db_folder_path):
    """
    Iterates through all folders in the Spider database directory,
    extracts the DDL from each .sqlite file, and saves it to ChromaDB.
    """
    # Initialize ChromaDB once before the loop
    chroma_client = chromadb.PersistentClient(path="./spider_rag_db")
    embedding_function = embedding_functions.DefaultEmbeddingFunction()
    schema_collection = chroma_client.get_or_create_collection(
        name="spider_schemas",
        embedding_function=embedding_function
    )
    
    # List all subdirectories inside the main "database" folder
    db_folders = [d for d in os.listdir(spider_db_folder_path) 
                  if os.path.isdir(os.path.join(spider_db_folder_path, d))]
    
    print(f"Found {len(db_folders)} databases. Starting extraction...")
    
    for db_id in db_folders:
        # In Spider, the sqlite file has the same name as its parent folder
        sqlite_path = os.path.join(spider_db_folder_path, db_id, f"{db_id}.sqlite")
        
        if os.path.exists(sqlite_path):
            # Step A: Extract
            ddl_content = extract_sqlite_ddl(sqlite_path)
            # Step B: Vectorize and Save
            setup_vector_database(db_id, ddl_content, schema_collection)
        else:
            print(f"WARNING: No .sqlite file found for {db_id} at {sqlite_path}")
            
    print("All databases have been successfully indexed into the RAG!")


In [10]:
# ==========================================
# 3. Execution
# ==========================================
# Replace this with the actual path where you extracted the Spider "database" folder
process_all_spider_databases("./spider_data/database")

Found 166 databases. Starting extraction...
Schema 'academic' vectorized successfully.
Schema 'activity_1' vectorized successfully.
Schema 'aircraft' vectorized successfully.
Schema 'allergy_1' vectorized successfully.
Schema 'apartment_rentals' vectorized successfully.
Schema 'architecture' vectorized successfully.
Schema 'assets_maintenance' vectorized successfully.
Schema 'baseball_1' vectorized successfully.
Schema 'battle_death' vectorized successfully.
Schema 'behavior_monitoring' vectorized successfully.
Schema 'bike_1' vectorized successfully.
Schema 'body_builder' vectorized successfully.
Schema 'book_2' vectorized successfully.
Schema 'browser_web' vectorized successfully.
Schema 'candidate_poll' vectorized successfully.
Schema 'car_1' vectorized successfully.
Schema 'chinook_1' vectorized successfully.
Schema 'cinema' vectorized successfully.
Schema 'city_record' vectorized successfully.
Schema 'climbing' vectorized successfully.
Schema 'club_1' vectorized successfully.
Sche

In [12]:
import os
import json
import csv
import chromadb
import google.generativeai as genai
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# ==========================================
# 1. API Configurations (Secure)
# ==========================================
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

# Google AI Studio Setup (Gemini)
genai.configure(api_key=google_api_key)

# Groq Setup (Llama 3 & Mistral)
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

# ChromaDB Setup (RAG) - Must exactly match Phase 2 paths
chroma_client = chromadb.PersistentClient(path="./spider_rag_db")
schema_collection = chroma_client.get_collection(name="spider_schemas")

# ==========================================
# 2. Helper Functions
# ==========================================
def retrieve_schema(db_id):
    """Retrieves the DDL schema from ChromaDB based on the database ID."""
    results = schema_collection.get(where={"db_id": db_id})
    if results['documents']:
        return results['documents'][0]
    return ""

def clean_sql_output(raw_sql):
    """Removes markdown formatting and line breaks to ensure a single-line SQL."""
    clean_str = raw_sql.replace("```sql", "").replace("```", "").strip()
    return clean_str.replace('\n', ' ')

# ==========================================
# 3. LLM Generators
# ==========================================
system_prompt = (
    "You are a Senior Data Engineer expert in SQLite. "
    "Given the database schema below, write the SQL query that answers the user's question. "
    "Return ONLY the valid SQL code, without markdown formatting, without explanations, and in a single line."
)

def generate_sql_gemini(question, ddl_schema):
    try:
        model = genai.GenerativeModel(
            model_name='gemini-2.5-flash',
            system_instruction=system_prompt
        )
        user_prompt = f"Database Schema:\n{ddl_schema}\n\nQuestion: {question}"
        response = model.generate_content(
            user_prompt,
            generation_config=genai.types.GenerationConfig(temperature=0.0)
        )
        return clean_sql_output(response.text)
    except Exception as e:
        return f"SELECT 'ERROR GEMINI: {e}'"

def generate_sql_groq(question, ddl_schema, model_id):
    try:
        user_prompt = f"Database Schema:\n{ddl_schema}\n\nQuestion: {question}"
        response = groq_client.chat.completions.create(
            model=model_id,
            temperature=0.0,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        return clean_sql_output(response.choices[0].message.content)
    except Exception as e:
        return f"SELECT 'ERROR GROQ: {e}'"

# ==========================================
# 4. Main Execution Pipeline
# ==========================================
def run_inference_pipeline(spider_json_path):
    """Runs the inference for all models and saves outputs to TXTs and a CSV."""
    
    # Load Spider dev.json
    with open(spider_json_path, 'r', encoding='utf-8') as f:
        spider_data = json.load(f)
        
    # Open text files for Spider evaluation
    f_gemini = open("predicted_gemini.txt", "w", encoding="utf-8")
    f_llama = open("predicted_llama.txt", "w", encoding="utf-8")
    f_mistral = open("predicted_mistral.txt", "w", encoding="utf-8")
    
    # Open CSV file for TCC analysis
    f_csv = open("results_comparison.csv", "w", encoding="utf-8", newline="")
    csv_writer = csv.writer(f_csv)
    csv_writer.writerow(["db_id", "question", "gold_sql", "gemini_sql", "llama_sql", "mistral_sql"])
    
    print(f"Starting inference for {len(spider_data)} questions...")
    
    for index, item in enumerate(spider_data):
        question = item['question']
        db_id = item['db_id']
        gold_sql = item['query']
        
        # 1. RAG Retrieval
        ddl_context = retrieve_schema(db_id)
        
        # 2. LLM Inference
        sql_gemini = generate_sql_gemini(question, ddl_context)
        sql_llama = generate_sql_groq(question, ddl_context, "llama3-70b-8192")
        sql_mistral = generate_sql_groq(question, ddl_context, "mixtral-8x7b-32768")
        
        # 3. Write to individual TXT files (For Phase 4)
        f_gemini.write(f"{sql_gemini}\n")
        f_llama.write(f"{sql_llama}\n")
        f_mistral.write(f"{sql_mistral}\n")
        
        # 4. Write to CSV (For analysis)
        csv_writer.writerow([db_id, question, gold_sql, sql_gemini, sql_llama, sql_mistral])
        
        # Progress tracker
        if (index + 1) % 10 == 0:
            print(f"Processed {index + 1} records...")

    # Close all files
    f_gemini.close()
    f_llama.close()
    f_mistral.close()
    f_csv.close()
    print("Pipeline finished successfully!")

# Example execution:
# run_inference_pipeline("spider/dev.json")

In [1]:
import os
import json
import csv
import time
import chromadb
import google.generativeai as genai
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Configurações de Caminhos
CHECKPOINT_CSV = "results_comparison.csv"
SPIDER_JSON = "spider_data/dev.json"

# ... (Funções retrieve_schema e LLM Generators permanecem as mesmas) ...

def get_last_processed_index():
    """Retorna quantos registros já foram salvos no CSV para retomar de onde parou."""
    if not os.path.exists(CHECKPOINT_CSV):
        return 0
    with open(CHECKPOINT_CSV, "r", encoding="utf-8") as f:
        # Subtrai 1 por causa do cabeçalho
        return sum(1 for line in f) - 1

def run_inference_pipeline(spider_json_path):
    with open(spider_json_path, 'r', encoding='utf-8') as f:
        spider_data = json.load(f)

    # 1. Verifica de onde retomar
    start_index = get_last_processed_index()
    
    if start_index >= len(spider_data):
        print("🎉 Todos os registros já foram processados!")
        return

    print(f"Retomando a partir do registro {start_index}...")

    # 2. Abre arquivos em modo APPEND ("a")
    # Se for o início (start_index == 0), escrevemos o cabeçalho no CSV
    write_header = not os.path.exists(CHECKPOINT_CSV)
    
    f_gemini = open("predicted_gemini.txt", "a", encoding="utf-8")
    f_llama = open("predicted_llama.txt", "a", encoding="utf-8")
    f_mistral = open("predicted_mistral.txt", "a", encoding="utf-8")
    f_csv = open(CHECKPOINT_CSV, "a", encoding="utf-8", newline="")
    csv_writer = csv.writer(f_csv)

    if write_header:
        csv_writer.writerow(["db_id", "question", "gold_sql", "gemini_sql", "llama_sql", "mistral_sql"])

    # 3. Loop começando do start_index
    try:
        for index in range(start_index, len(spider_data)):
            item = spider_data[index]
            question = item['question']
            db_id = item['db_id']
            gold_sql = item['query']

            ddl_context = retrieve_schema(db_id)

            # Inferência com tratamento de erro e retry interno (como vimos antes)
            sql_gemini = generate_sql_gemini(question, ddl_context)
            sql_llama = generate_sql_groq(question, ddl_context, "llama-3.3-70b-versatile")
            sql_mistral = generate_sql_groq(question, ddl_context, "llama-3.1-8b-instant")

            f_gemini.write(f"{sql_gemini}\n")
            f_llama.write(f"{sql_llama}\n")
            f_mistral.write(f"{sql_mistral}\n")
            csv_writer.writerow([db_id, question, gold_sql, sql_gemini, sql_llama, sql_mistral])
            
            # Flush garante que o dado seja escrito no disco imediatamente
            f_csv.flush() 

            print(f"[{index+1}/{len(spider_data)}] Processado: {db_id}")
            time.sleep(12) # Pausa maior para evitar o erro 429 rapidamente

    except KeyboardInterrupt:
        print("\nInterrompido pelo usuário. Progresso salvo com segurança.")
    finally:
        f_gemini.close()
        f_llama.close()
        f_mistral.close()
        f_csv.close()

# Executar
# run_inference_pipeline(SPIDER_JSON)

d:\Pos AI gen\TCC\llm-sql\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\nicol\AppData\Local\Temp\ipykernel_28764\2059723050.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
import os
import json
import csv
import time
import chromadb
import google.generativeai as genai
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# ==========================================
# 1. Configuration & Paths
# ==========================================
# API Keys
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

# File Paths
SPIDER_JSON = "spider_data/dev.json"
CHECKPOINT_CSV = "results_comparison.csv"

# API Setup
genai.configure(api_key=google_api_key)
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

# ChromaDB Setup (RAG)
chroma_client = chromadb.PersistentClient(path="./spider_rag_db")
schema_collection = chroma_client.get_collection(name="spider_schemas")

# ==========================================
# 2. Helper Functions
# ==========================================
def retrieve_schema(db_id):
    """Retrieves the DDL schema from ChromaDB based on the database ID."""
    results = schema_collection.get(where={"db_id": db_id})
    if results['documents']:
        return results['documents'][0]
    return ""

def clean_sql_output(raw_sql):
    """Removes markdown formatting and line breaks to ensure a single-line SQL."""
    clean_str = raw_sql.replace("```sql", "").replace("```", "").strip()
    return clean_str.replace('\n', ' ')

def get_last_processed_index():
    """Returns how many records have already been saved to the CSV to resume progress."""
    if not os.path.exists(CHECKPOINT_CSV):
        return 0
    with open(CHECKPOINT_CSV, "r", encoding="utf-8") as f:
        # Subtract 1 because of the header row
        return sum(1 for line in f) - 1

# ==========================================
# 3. LLM Generators
# ==========================================
system_prompt = (
    "You are a Senior Data Engineer expert in SQLite. "
    "Given the database schema below, write the SQL query that answers the user's question. "
    "Return ONLY the valid SQL code, without markdown formatting, without explanations, and in a single line."
)

def generate_sql_gemini(question, ddl_schema):
    try:
        model = genai.GenerativeModel(
            model_name='gemini-3.1-flash-lite-preview',
            system_instruction=system_prompt
        )
        user_prompt = f"Database Schema:\n{ddl_schema}\n\nQuestion: {question}"
        response = model.generate_content(
            user_prompt,
            generation_config=genai.types.GenerationConfig(temperature=0.0)
        )
        return clean_sql_output(response.text)
    except Exception as e:
        error_msg = str(e)
        # Check for Rate Limit (429) or Quota errors
        if "429" in error_msg or "Quota" in error_msg:
            print("\n[!] Gemini rate limit reached. Sleeping for 60 seconds...")
            time.sleep(60)
            return generate_sql_gemini(question, ddl_schema) # Recursive retry
            
        return f"SELECT 'ERROR GEMINI: {e}'"

def generate_sql_groq(question, ddl_schema, model_id):
    try:
        user_prompt = f"Database Schema:\n{ddl_schema}\n\nQuestion: {question}"
        response = groq_client.chat.completions.create(
            model=model_id,
            temperature=0.0,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        return clean_sql_output(response.choices[0].message.content)
    except Exception as e:
        return f"SELECT 'ERROR GROQ: {e}'"

# ==========================================
# 4. Main Execution Pipeline
# ==========================================
def run_inference_pipeline(spider_json_path):
    """Runs the inference with checkpointing to resume safely after interruptions."""
    
    with open(spider_json_path, 'r', encoding='utf-8') as f:
        spider_data = json.load(f)
        
    start_index = get_last_processed_index()
    
    if start_index >= len(spider_data):
        print("🎉 All records have already been processed!")
        return

    print(f"Resuming inference from record {start_index} out of {len(spider_data)}...")
    
    # Open files in APPEND mode ("a") to add to the end of the file
    write_header = not os.path.exists(CHECKPOINT_CSV)
    
    f_gemini = open("predicted_gemini.txt", "a", encoding="utf-8")
    f_llama = open("predicted_llama.txt", "a", encoding="utf-8")
    f_qwen = open("predicted_qwen.txt", "a", encoding="utf-8")
    
    f_csv = open(CHECKPOINT_CSV, "a", encoding="utf-8", newline="")
    csv_writer = csv.writer(f_csv)
    
    if write_header:
        csv_writer.writerow(["db_id", "question", "gold_sql", "gemini_sql", "llama_sql", "qwen_sql"])
        
    try:
        for index in range(start_index, len(spider_data)):
            item = spider_data[index]
            question = item['question']
            db_id = item['db_id']
            gold_sql = item['query']
            
            # 1. RAG Retrieval
            ddl_context = retrieve_schema(db_id)
            
            # 2. LLM Inference
            sql_gemini = generate_sql_gemini(question, ddl_context)
            sql_llama = generate_sql_groq(question, ddl_context, "llama-3.3-70b-versatile")
            sql_qwen = generate_sql_groq(question, ddl_context, "qwen/qwen3-32b")
            
            # 3. Save to individual TXT files
            f_gemini.write(f"{sql_gemini}\n")
            f_llama.write(f"{sql_llama}\n")
            f_qwen.write(f"{sql_qwen}\n")
            
            # 4. Save to CSV and flush to disk immediately
            csv_writer.writerow([db_id, question, gold_sql, sql_gemini, sql_llama, sql_qwen])
            f_csv.flush()
            
            print(f"[{index + 1}/{len(spider_data)}] Processed successfully: {db_id}")
            
            # Throttle to respect the Gemini Free Tier limit (approx 5 requests/min)
            time.sleep(12) 

    except KeyboardInterrupt:
        print("\n[!] Execution interrupted by the user. Progress safely saved.")
    finally:
        f_gemini.close()
        f_llama.close()
        f_qwen.close()
        f_csv.close()
        print("Pipeline shutdown complete.")

# Run the pipeline
if __name__ == "__main__":
    run_inference_pipeline(SPIDER_JSON)

Resuming inference from record 0 out of 1034...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...

[!] Gemini rate limit reached. Sleeping for 60 seconds...


In [5]:
# Executar
run_inference_pipeline(SPIDER_JSON)

Resuming inference from record 0 out of 1034...
[1/1034] Processed successfully: concert_singer
[2/1034] Processed successfully: concert_singer
[3/1034] Processed successfully: concert_singer
[4/1034] Processed successfully: concert_singer
[5/1034] Processed successfully: concert_singer
[6/1034] Processed successfully: concert_singer
[7/1034] Processed successfully: concert_singer
[8/1034] Processed successfully: concert_singer
[9/1034] Processed successfully: concert_singer
[10/1034] Processed successfully: concert_singer
[11/1034] Processed successfully: concert_singer
[12/1034] Processed successfully: concert_singer
[13/1034] Processed successfully: concert_singer
[14/1034] Processed successfully: concert_singer
[15/1034] Processed successfully: concert_singer
[16/1034] Processed successfully: concert_singer

[!] Execution interrupted by the user. Progress safely saved.
Pipeline shutdown complete.


In [7]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

print("Buscando modelos ativos na Groq...\n")
modelos = groq_client.models.list()

for modelo in modelos.data:
    # Vamos filtrar para mostrar todos, mas destacar os Qwen
    if "qwen" in modelo.id.lower():
        print(f"✅ MODELO QWEN DISPONÍVEL: '{modelo.id}'")
    else:
        print(f"- {modelo.id}")

Buscando modelos ativos na Groq...

- canopylabs/orpheus-arabic-saudi
- moonshotai/kimi-k2-instruct-0905
- openai/gpt-oss-20b
- openai/gpt-oss-safeguard-20b
- whisper-large-v3-turbo
- whisper-large-v3
- llama-3.3-70b-versatile
- allam-2-7b
- moonshotai/kimi-k2-instruct
- meta-llama/llama-prompt-guard-2-22m
- groq/compound-mini
- meta-llama/llama-4-scout-17b-16e-instruct
- groq/compound
- openai/gpt-oss-120b
✅ MODELO QWEN DISPONÍVEL: 'qwen/qwen3-32b'
- llama-3.1-8b-instant
- meta-llama/llama-prompt-guard-2-86m
- canopylabs/orpheus-v1-english


In [13]:
run_inference_pipeline("spider_data/dev.json")

Starting inference for 1034 questions...
Processed 10 records...
Processed 20 records...
Processed 30 records...
Processed 40 records...
Processed 50 records...
Processed 60 records...
Processed 70 records...
Processed 80 records...
Processed 90 records...
Processed 100 records...
Processed 110 records...
Processed 120 records...
Processed 130 records...
Processed 140 records...
Processed 150 records...
Processed 160 records...
Processed 170 records...
Processed 180 records...
Processed 190 records...
Processed 200 records...
Processed 210 records...
Processed 220 records...
Processed 230 records...
Processed 240 records...
Processed 250 records...
Processed 260 records...
Processed 270 records...
Processed 280 records...
Processed 290 records...
Processed 300 records...
Processed 310 records...
Processed 320 records...
Processed 330 records...
Processed 340 records...
Processed 350 records...
Processed 360 records...
Processed 370 records...
Processed 380 records...
Processed 390 reco